In [ ]:
# CELL 1 - IMPORT LIBRARY & LOAD DATASET


from pathlib import Path
import pandas as pd
import numpy as np

# 1. KONFIGURASI PATH
BASE_DIR = Path(
    r"D:\1-Kuliah\SEKRIPSI\Skripsi_Traffic_Density_Estimation_and_Forecasting"
)

DATA_DIR = BASE_DIR /"04_Vehicle_Counting"/ "final-dataset-vehicle-counting"

INPUT_PATH = DATA_DIR / "Semua_6_sampai_20_April_interval_counting.csv"

OUTPUT_DIR = BASE_DIR / "05_Traffic_Analysis" / "output_mkji_convert_directory"
OUTPUT_PATH_FULL = OUTPUT_DIR / "dataset_mkji.csv"

# 2. CEK FILE INPUT

print("Dataset ditemukan:")
print(INPUT_PATH)



# 3. LOAD DATASET


df = pd.read_csv(INPUT_PATH)

print("\nDataset berhasil dimuat.")
print(f"Jumlah baris  : {len(df)}")
print(f"Jumlah kolom  : {len(df.columns)}")



# 4. TAMPILKAN INFORMASI DATASET


print("\nDaftar kolom:")
print(df.columns.tolist())

print("\n5 data pertama:")
display(df.head())

Dataset ditemukan:
D:\1-Kuliah\SEKRIPSI\Skripsi_Traffic_Density_Estimation_and_Forecasting\04_Vehicle_Counting\final-dataset-vehicle-counting\Semua_6_sampai_20_April_interval_counting.csv

Dataset berhasil dimuat.
Jumlah baris  : 704
Jumlah kolom  : 13

Daftar kolom:
['Date', 'Time', 'session', 'day_type', 'motor_IN', 'mobil_IN', 'bus_IN', 'truk_IN', 'motor_OUT', 'mobil_OUT', 'bus_OUT', 'truk_OUT', 'Source_File']

5 data pertama:


,Date,Time,session,day_type,motor_IN,mobil_IN,bus_IN,truk_IN,motor_OUT,mobil_OUT,bus_OUT,truk_OUT,Source_File
0,4/6/2026,6:00:00,pagi,0,924,284,5,29,555,235,4,14,Pagi_Senin6_April_interval_counting.csv
1,4/6/2026,6:15:00,pagi,0,1386,458,6,13,933,397,6,10,Pagi_Senin6_April_interval_counting.csv
2,4/6/2026,6:30:00,pagi,0,1826,541,8,18,1291,446,2,3,Pagi_Senin6_April_interval_counting.csv
3,4/6/2026,6:45:00,pagi,0,2229,541,7,12,1298,398,5,6,Pagi_Senin6_April_interval_counting.csv
4,4/6/2026,7:00:00,pagi,0,2354,536,0,26,1432,425,0,11,Pagi_Senin6_April_interval_counting.csv


In [ ]:

# CELL 2 - VALIDASI & PERSIAPAN DATA



# 1. DAFTAR KOLOM YANG DIBUTUHKAN


vehicle_columns = [
    "motor_IN",
    "mobil_IN",
    "bus_IN",
    "truk_IN",
    "motor_OUT",
    "mobil_OUT",
    "bus_OUT",
    "truk_OUT"
]

required_columns = [
    "Date",
    "Time",
    "session",
    "day_type"
] + vehicle_columns



# 2. CEK KELENGKAPAN KOLOM


missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Kolom berikut tidak ditemukan: {missing_columns}"
    )

print("Semua kolom yang dibutuhkan tersedia.")



# 3. KONVERSI DATE & TIME


df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

df["Time"] = pd.to_timedelta(
    df["Time"],
    errors="coerce"
)



# 4. KONVERSI KOLOM KENDARAAN KE NUMERIK


for col in vehicle_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )



# 5. CEK MISSING VALUE


print("\nMissing value:")
print(df[required_columns].isnull().sum())



# 6. CEK NILAI NEGATIF


negative_count = (df[vehicle_columns] < 0).sum()

print("\nJumlah nilai negatif:")
print(negative_count)



# 7. CEK DAY TYPE & SESSION


print("\nNilai unik day_type:")
print(sorted(df["day_type"].dropna().unique()))

print("\nNilai unik session:")
print(df["session"].dropna().unique())



# 8. URUTKAN DATA


df = (
    df
    .sort_values(["Date", "Time"])
    .reset_index(drop=True)
)



# 9. TAMPILKAN HASIL


print("\nJumlah data:", len(df))

display(
    df[
        [
            "Date",
            "Time",
            "session",
            "day_type"
        ] + vehicle_columns
    ].head()
)

Semua kolom yang dibutuhkan tersedia.

Missing value:
Date         0
Time         0
session      0
day_type     0
motor_IN     0
mobil_IN     0
bus_IN       0
truk_IN      0
motor_OUT    0
mobil_OUT    0
bus_OUT      0
truk_OUT     0
dtype: int64

Jumlah nilai negatif:
motor_IN     0
mobil_IN     0
bus_IN       0
truk_IN      0
motor_OUT    0
mobil_OUT    0
bus_OUT      0
truk_OUT     0
dtype: int64

Nilai unik day_type:
[np.int64(0), np.int64(1)]

Nilai unik session:
['pagi' 'sore']

Jumlah data: 704


,Date,Time,session,day_type,motor_IN,mobil_IN,bus_IN,truk_IN,motor_OUT,mobil_OUT,bus_OUT,truk_OUT
0,2026-04-06,0 days 06:00:00,pagi,0,924,284,5,29,555,235,4,14
1,2026-04-06,0 days 06:15:00,pagi,0,1386,458,6,13,933,397,6,10
2,2026-04-06,0 days 06:30:00,pagi,0,1826,541,8,18,1291,446,2,3
3,2026-04-06,0 days 06:45:00,pagi,0,2229,541,7,12,1298,398,5,6
4,2026-04-06,0 days 07:00:00,pagi,0,2354,536,0,26,1432,425,0,11


## Menghitung SMP dan Q

In [ ]:

# CELL 3 - KONVERSI KENDARAAN KE SMP & HITUNG Q



# 1. PARAMETER EMP


EMP_MC = 0.25   # Sepeda motor
EMP_LV = 1.00   # Kendaraan ringan
EMP_HV = 1.20   # Kendaraan berat (bus + truk)

INTERVAL_MINUTES = 15



# 2. HITUNG KELOMPOK KENDARAAN


# MC = Motorcycle
df["MC_IN"] = df["motor_IN"]
df["MC_OUT"] = df["motor_OUT"]

# LV = Light Vehicle
df["LV_IN"] = df["mobil_IN"]
df["LV_OUT"] = df["mobil_OUT"]

# HV = Heavy Vehicle
df["HV_IN"] = df["bus_IN"] + df["truk_IN"]
df["HV_OUT"] = df["bus_OUT"] + df["truk_OUT"]



# 3. KONVERSI KENDARAAN KE SMP PER 15 MENIT


df["Q_IN_15min"] = (
    (df["MC_IN"] * EMP_MC)
    + (df["LV_IN"] * EMP_LV)
    + (df["HV_IN"] * EMP_HV)
)

df["Q_OUT_15min"] = (
    (df["MC_OUT"] * EMP_MC)
    + (df["LV_OUT"] * EMP_LV)
    + (df["HV_OUT"] * EMP_HV)
)



# 4. KONVERSI SMP/15 MENIT MENJADI Q (SMP/JAM)


conversion_factor = 60 / INTERVAL_MINUTES

df["Q_IN"] = df["Q_IN_15min"] * conversion_factor
df["Q_OUT"] = df["Q_OUT_15min"] * conversion_factor



# 5. TAMPILKAN HASIL


print(f"Interval            : {INTERVAL_MINUTES} menit")
print(f"Faktor konversi Q   : x{conversion_factor:.0f}")

print("\nParameter EMP:")
print(f"MC (motor)      = {EMP_MC}")
print(f"LV (mobil)      = {EMP_LV}")
print(f"HV (bus + truk) = {EMP_HV}")

display(
    df[
        [
            "Date",
            "Time",
            "session",
            "MC_IN",
            "LV_IN",
            "HV_IN",
            "MC_OUT",
            "LV_OUT",
            "HV_OUT",
            "Q_IN_15min",
            "Q_OUT_15min",
            "Q_IN",
            "Q_OUT"
        ]
    ].head()
)

Interval            : 15 menit
Faktor konversi Q   : x4

Parameter EMP:
MC (motor)      = 0.25
LV (mobil)      = 1.0
HV (bus + truk) = 1.2


,Date,Time,session,MC_IN,LV_IN,HV_IN,MC_OUT,LV_OUT,HV_OUT,Q_IN_15min,Q_OUT_15min,Q_IN,Q_OUT
0,2026-04-06,0 days 06:00:00,pagi,924,284,34,555,235,18,555.80,395.35,2223.2,1581.4
1,2026-04-06,0 days 06:15:00,pagi,1386,458,19,933,397,16,827.30,649.45,3309.2,2597.8
2,2026-04-06,0 days 06:30:00,pagi,1826,541,26,1291,446,5,1028.70,774.75,4114.8,3099.0
3,2026-04-06,0 days 06:45:00,pagi,2229,541,19,1298,398,11,1121.05,735.70,4484.2,2942.8
4,2026-04-06,0 days 07:00:00,pagi,2354,536,26,1432,425,11,1155.70,796.20,4622.8,3184.8


## Menghitung KAPASITAS (C) & DERAJAT KEJENUHAN (DS)

In [ ]:

# CELL 4 - HITUNG KAPASITAS (C) & DERAJAT KEJENUHAN (DS)



# 1. KAPASITAS DASAR (C0)


C0_PER_LAJUR = 1650  # smp/jam/lajur

JUMLAH_LAJUR_IN = 3
JUMLAH_LAJUR_OUT = 5

C0_IN = C0_PER_LAJUR * JUMLAH_LAJUR_IN
C0_OUT = C0_PER_LAJUR * JUMLAH_LAJUR_OUT



# 2. FAKTOR PENYESUAIAN KAPASITAS


FCW = 1.00
FCSP = 1.00
FCSF = 0.968
FCCS = 1.00



# 3. HITUNG KAPASITAS JALAN


C_IN = (
    C0_IN
    * FCW
    * FCSP
    * FCSF
    * FCCS
)

C_OUT = (
    C0_OUT
    * FCW
    * FCSP
    * FCSF
    * FCCS
)



# 4. SIMPAN KAPASITAS KE DATAFRAME


df["C_IN"] = C_IN
df["C_OUT"] = C_OUT



# 5. HITUNG DERAJAT KEJENUHAN (DS)


df["DS_IN"] = df["Q_IN"] / df["C_IN"]
df["DS_OUT"] = df["Q_OUT"] / df["C_OUT"]



# 6. TAMPILKAN PARAMETER KAPASITAS


print("=== PARAMETER KAPASITAS ===")

print(f"C0 IN  : {C0_IN:.2f} smp/jam")
print(f"C0 OUT : {C0_OUT:.2f} smp/jam")

print(f"\nFCW  : {FCW}")
print(f"FCSP : {FCSP}")
print(f"FCSF : {FCSF}")
print(f"FCCS : {FCCS}")

print("\n=== KAPASITAS AKHIR ===")

print(f"C IN  : {C_IN:.2f} smp/jam")
print(f"C OUT : {C_OUT:.2f} smp/jam")



# 7. TAMPILKAN HASIL


display(
    df[
        [
            "Date",
            "Time",
            "session",
            "Q_IN_15min",
            "Q_OUT_15min",
            "Q_IN",
            "Q_OUT",
            "C_IN",
            "C_OUT",
            "DS_IN",
            "DS_OUT"
        ]
    ].head()
)

=== PARAMETER KAPASITAS ===
C0 IN  : 4950.00 smp/jam
C0 OUT : 8250.00 smp/jam

FCW  : 1.0
FCSP : 1.0
FCSF : 0.968
FCCS : 1.0

=== KAPASITAS AKHIR ===
C IN  : 4791.60 smp/jam
C OUT : 7986.00 smp/jam


,Date,Time,session,Q_IN_15min,Q_OUT_15min,Q_IN,Q_OUT,C_IN,C_OUT,DS_IN,DS_OUT
0,2026-04-06,0 days 06:00:00,pagi,555.80,395.35,2223.2,1581.4,4791.6,7986.0,0.463979,0.198022
1,2026-04-06,0 days 06:15:00,pagi,827.30,649.45,3309.2,2597.8,4791.6,7986.0,0.690625,0.325294
2,2026-04-06,0 days 06:30:00,pagi,1028.70,774.75,4114.8,3099.0,4791.6,7986.0,0.858753,0.388054
3,2026-04-06,0 days 06:45:00,pagi,1121.05,735.70,4484.2,2942.8,4791.6,7986.0,0.935846,0.368495
4,2026-04-06,0 days 07:00:00,pagi,1155.70,796.20,4622.8,3184.8,4791.6,7986.0,0.964772,0.398798


## Cell 5 — Konversi DS ke LOS

In [ ]:

# CELL 5 - KONVERSI DERAJAT KEJENUHAN (DS) KE LOS



# 1. FUNGSI KLASIFIKASI LOS


def classify_los(ds):
    if pd.isna(ds):
        return np.nan

    # Pembulatan mengikuti interval pada tabel referensi
    ds_rounded = round(ds, 2)

    if ds_rounded < 0.04:
        return "A"
    elif ds_rounded <= 0.24:
        return "B"
    elif ds_rounded <= 0.54:
        return "C"
    elif ds_rounded <= 0.80:
        return "D"
    elif ds_rounded <= 1.00:
        return "E"
    else:
        return "F"



# 2. KETERANGAN KONDISI LALU LINTAS


los_description = {
    "A": "Arus lalu lintas tenang, kendaraan dapat bergerak dengan kecepatan bebas.",
    "B": "Arus mulai sedikit ramai dan kecepatan kendaraan mulai menurun.",
    "C": "Arus lalu lintas padat dan pergerakan kendaraan mulai dibatasi.",
    "D": "Arus lalu lintas mendekati kondisi jenuh dan kecepatan mulai rendah.",
    "E": "Arus mulai mengalami kemacetan dan kecepatan kendaraan rendah.",
    "F": "Kondisi lalu lintas macet dengan kecepatan kendaraan sangat rendah."
}



# 3. KONVERSI DS MENJADI LOS


df["LOS_IN"] = df["DS_IN"].apply(classify_los)
df["LOS_OUT"] = df["DS_OUT"].apply(classify_los)



# 4. TAMBAHKAN KETERANGAN KONDISI


df["Kondisi_IN"] = df["LOS_IN"].map(los_description)
df["Kondisi_OUT"] = df["LOS_OUT"].map(los_description)



# 5. TAMPILKAN HASIL


display(
    df[
        [
            "Date",
            "Time",
            "session",
            "Q_IN",
            "C_IN",
            "DS_IN",
            "LOS_IN",
            "Kondisi_IN",
            "Q_OUT",
            "C_OUT",
            "DS_OUT",
            "LOS_OUT",
            "Kondisi_OUT"
        ]
    ].head(10)
)

,Date,Time,session,Q_IN,C_IN,DS_IN,LOS_IN,Kondisi_IN,Q_OUT,C_OUT,DS_OUT,LOS_OUT,Kondisi_OUT
0,2026-04-06,0 days 06:00:00,pagi,2223.2,4791.6,0.463979,C,Arus lalu lintas padat dan pergerakan kendaraa...,1581.4,7986.0,0.198022,B,Arus mulai sedikit ramai dan kecepatan kendara...
1,2026-04-06,0 days 06:15:00,pagi,3309.2,4791.6,0.690625,D,Arus lalu lintas mendekati kondisi jenuh dan k...,2597.8,7986.0,0.325294,C,Arus lalu lintas padat dan pergerakan kendaraa...
2,2026-04-06,0 days 06:30:00,pagi,4114.8,4791.6,0.858753,E,Arus mulai mengalami kemacetan dan kecepatan k...,3099.0,7986.0,0.388054,C,Arus lalu lintas padat dan pergerakan kendaraa...
3,2026-04-06,0 days 06:45:00,pagi,4484.2,4791.6,0.935846,E,Arus mulai mengalami kemacetan dan kecepatan k...,2942.8,7986.0,0.368495,C,Arus lalu lintas padat dan pergerakan kendaraa...
4,2026-04-06,0 days 07:00:00,pagi,4622.8,4791.6,0.964772,E,Arus mulai mengalami kemacetan dan kecepatan k...,3184.8,7986.0,0.398798,C,Arus lalu lintas padat dan pergerakan kendaraa...
5,2026-04-06,0 days 07:15:00,pagi,4543.0,4791.6,0.948118,E,Arus mulai mengalami kemacetan dan kecepatan k...,2614.0,7986.0,0.327323,C,Arus lalu lintas padat dan pergerakan kendaraa...
6,2026-04-06,0 days 07:30:00,pagi,4477.4,4791.6,0.934427,E,Arus mulai mengalami kemacetan dan kecepatan k...,2719.0,7986.0,0.340471,C,Arus lalu lintas padat dan pergerakan kendaraa...
7,2026-04-06,0 days 07:45:00,pagi,3821.0,4791.6,0.797437,D,Arus lalu lintas mendekati kondisi jenuh dan k...,2530.8,7986.0,0.316905,C,Arus lalu lintas padat dan pergerakan kendaraa...
8,2026-04-06,0 days 08:00:00,pagi,3549.2,4791.6,0.740713,D,Arus lalu lintas mendekati kondisi jenuh dan k...,2460.4,7986.0,0.308089,C,Arus lalu lintas padat dan pergerakan kendaraa...
9,2026-04-06,0 days 08:15:00,pagi,3345.8,4791.6,0.698264,D,Arus lalu lintas mendekati kondisi jenuh dan k...,2215.8,7986.0,0.277461,C,Arus lalu lintas padat dan pergerakan kendaraa...


In [ ]:

# CELL 6 - FINALISASI & SAVE DATASET MKJI


# Rename nama kolom agar konsisten berbahasa indonesia
column_mapping = {
    "Date": "tanggal",
    "Time": "waktu",
    "session": "sesi",
    "day_type": "tipe_hari"
}
df = df.rename(columns=column_mapping)


# 1. TENTUKAN URUTAN KOLOM FINAL


final_columns = [
    # Informasi waktu
    "tanggal",
    "waktu",
    "sesi",
    "tipe_hari",

    # # Data kendaraan asli - IN
    # "motor_IN",
    # "mobil_IN",
    # "bus_IN",
    # "truk_IN",

    # # Data kendaraan asli - OUT
    # "motor_OUT",
    # "mobil_OUT",
    # "bus_OUT",
    # "truk_OUT",

    # Klasifikasi kendaraan MKJI
    "MC_IN",
    "LV_IN",
    "HV_IN",
    "MC_OUT",
    "LV_OUT",
    "HV_OUT",

    # Volume lalu lintas
    "Q_IN_15min",
    "Q_OUT_15min",
    "Q_IN",
    "Q_OUT",

    # Kapasitas jalan
    "C_IN",
    "C_OUT",

    # Derajat kejenuhan
    "DS_IN",
    "DS_OUT",

    # Level of Service
    "LOS_IN",
    "LOS_OUT",

    # Kondisi lalu lintas
    "Kondisi_IN",
    "Kondisi_OUT"
]



# 2. BUAT DATASET FINAL


df_final = df[final_columns].copy()



# 3. URUTKAN BERDASARKAN TANGGAL & WAKTU


df_final = (
    df_final
    .sort_values(["tanggal", "waktu"])
    .reset_index(drop=True)
)



# 4. FORMAT DATE & TIME


df_final["tanggal"] = df_final["tanggal"].dt.strftime("%Y-%m-%d")

df_final["waktu"] = (
    df_final["waktu"]
    .astype(str)
    .str.replace("0 days ", "", regex=False)
)


# 5. SIMPAN DATASET


# full dengan MC, LV, HV, Q, C, DS, LOS, dan Kondisi
df_final.to_csv(
    OUTPUT_PATH_FULL,
    index=False,
    encoding="utf-8-sig"
)



# KONFIRMASI HASIL


print("Dataset MKJI berhasil disimpan.")
print(f"Lokasi       : {OUTPUT_PATH_FULL}")
print(f"Jumlah baris : {len(df_final)}")
print(f"Jumlah kolom : {len(df_final.columns)}")

print("\nPreview dataset final:")
display(df_final.head())


Dataset MKJI berhasil disimpan.
Lokasi       : D:\1-Kuliah\SEKRIPSI\Skripsi_Traffic_Density_Estimation_and_Forecasting\05_Traffic_Analysis\output_mkji_convert_directory\dataset_mkji.csv
Jumlah baris : 704
Jumlah kolom : 22

Preview dataset final:


,tanggal,waktu,sesi,tipe_hari,MC_IN,LV_IN,HV_IN,MC_OUT,LV_OUT,HV_OUT,...,Q_IN,Q_OUT,C_IN,C_OUT,DS_IN,DS_OUT,LOS_IN,LOS_OUT,Kondisi_IN,Kondisi_OUT
0,2026-04-06,06:00:00,pagi,0,924,284,34,555,235,18,...,2223.2,1581.4,4791.6,7986.0,0.463979,0.198022,C,B,Arus lalu lintas padat dan pergerakan kendaraa...,Arus mulai sedikit ramai dan kecepatan kendara...
1,2026-04-06,06:15:00,pagi,0,1386,458,19,933,397,16,...,3309.2,2597.8,4791.6,7986.0,0.690625,0.325294,D,C,Arus lalu lintas mendekati kondisi jenuh dan k...,Arus lalu lintas padat dan pergerakan kendaraa...
2,2026-04-06,06:30:00,pagi,0,1826,541,26,1291,446,5,...,4114.8,3099.0,4791.6,7986.0,0.858753,0.388054,E,C,Arus mulai mengalami kemacetan dan kecepatan k...,Arus lalu lintas padat dan pergerakan kendaraa...
3,2026-04-06,06:45:00,pagi,0,2229,541,19,1298,398,11,...,4484.2,2942.8,4791.6,7986.0,0.935846,0.368495,E,C,Arus mulai mengalami kemacetan dan kecepatan k...,Arus lalu lintas padat dan pergerakan kendaraa...
4,2026-04-06,07:00:00,pagi,0,2354,536,26,1432,425,11,...,4622.8,3184.8,4791.6,7986.0,0.964772,0.398798,E,C,Arus mulai mengalami kemacetan dan kecepatan k...,Arus lalu lintas padat dan pergerakan kendaraa...


In [9]:
# Set agar SEMUA 22 kolom ditampilkan ke samping
pd.set_option("display.max_columns", None)

# Set agar SEMUA baris ditampilkan ke bawah
# pd.set_option("display.max_rows", None)

# Atur lebar tampilan teks agar kolom tidak terpotong atau terlipat
pd.set_option("display.width", 1000)

display(df_final.head())

,tanggal,waktu,sesi,tipe_hari,MC_IN,LV_IN,HV_IN,MC_OUT,LV_OUT,HV_OUT,Q_IN_15min,Q_OUT_15min,Q_IN,Q_OUT,C_IN,C_OUT,DS_IN,DS_OUT,LOS_IN,LOS_OUT,Kondisi_IN,Kondisi_OUT
0,2026-04-06,06:00:00,pagi,0,924,284,34,555,235,18,555.80,395.35,2223.2,1581.4,4791.6,7986.0,0.463979,0.198022,C,B,Arus lalu lintas padat dan pergerakan kendaraa...,Arus mulai sedikit ramai dan kecepatan kendara...
1,2026-04-06,06:15:00,pagi,0,1386,458,19,933,397,16,827.30,649.45,3309.2,2597.8,4791.6,7986.0,0.690625,0.325294,D,C,Arus lalu lintas mendekati kondisi jenuh dan k...,Arus lalu lintas padat dan pergerakan kendaraa...
2,2026-04-06,06:30:00,pagi,0,1826,541,26,1291,446,5,1028.70,774.75,4114.8,3099.0,4791.6,7986.0,0.858753,0.388054,E,C,Arus mulai mengalami kemacetan dan kecepatan k...,Arus lalu lintas padat dan pergerakan kendaraa...
3,2026-04-06,06:45:00,pagi,0,2229,541,19,1298,398,11,1121.05,735.70,4484.2,2942.8,4791.6,7986.0,0.935846,0.368495,E,C,Arus mulai mengalami kemacetan dan kecepatan k...,Arus lalu lintas padat dan pergerakan kendaraa...
4,2026-04-06,07:00:00,pagi,0,2354,536,26,1432,425,11,1155.70,796.20,4622.8,3184.8,4791.6,7986.0,0.964772,0.398798,E,C,Arus mulai mengalami kemacetan dan kecepatan k...,Arus lalu lintas padat dan pergerakan kendaraa...


In [ ]:
# # full dengan MC, LV, HV, Q, C, DS, LOS, dan Kondisi

# final_columns_clean = [
# # nanti isi dengan kolom yang ingin disimpan di dataset

# ]
# df_final.to_csv(
#     OUTPUT_PATH_CLEAN,
#     index=False,
#     encoding="utf-8-sig"
# )


# 
# # KONFIRMASI HASIL
# 

# print("Dataset MKJI berhasil disimpan.")
# print(f"Lokasi       : {OUTPUT_PATH_CLEAN}")
# print(f"Jumlah baris : {len(df_final)}")
# print(f"Jumlah kolom : {len(df_final.columns)}")

# print("\nPreview dataset final:")
# display(df_final.head())


